# CLAP — Music Similarity via Language-Audio Embeddings

CLAP (Contrastive Language-Audio Pretraining) learns a shared embedding space for audio and text. This enables:
- **Audio-to-audio similarity**: Find similar songs
- **Text-to-audio search**: 'Find me upbeat jazz piano'
- **Audio-to-text description**: What does this sound like?

This is the technology behind modern music recommendation and search.

In [ ]:
!pip install transformers torch torchaudio librosa matplotlib numpy scipy IPython

In [ ]:
from transformers import ClapProcessor, ClapModel
import torch

processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
model = ClapModel.from_pretrained("laion/clap-htsat-unfused")
model.eval()
print("CLAP model loaded")

## Text-to-Audio Similarity

Compute how well text descriptions match audio clips.

In [ ]:
import librosa
import numpy as np

# Create simple synthetic audio samples
sr = 48000
duration = 5

# Sample 1: Sine tone (simple, pure)
t = np.linspace(0, duration, sr * duration)
audio_sine = np.sin(2 * np.pi * 440 * t).astype(np.float32) * 0.5

# Sample 2: Drum-like (noise burst)
audio_drum = np.zeros(sr * duration, dtype=np.float32)
for i in range(0, sr * duration, sr // 2):
    burst = np.random.randn(sr // 10).astype(np.float32) * 0.5
    env = np.exp(-np.linspace(0, 5, len(burst)))
    audio_drum[i:i+len(burst)] += burst * env

# Sample 3: Chord (harmonics)
audio_chord = sum(np.sin(2*np.pi*f*t) for f in [261.6, 329.6, 392.0]).astype(np.float32) * 0.3

samples = {"Sine tone": audio_sine, "Drum pattern": audio_drum, "Major chord": audio_chord}
descriptions = ["A pure sine wave tone", "Percussion and drum beats", "A major chord on piano",
                "Energetic dance music", "Calm ambient soundscape"]

# Compute similarities
for name, audio in samples.items():
    inputs = processor(text=descriptions, audios=[audio], sampling_rate=sr, 
                      return_tensors="pt", padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits_per_audio[0]
    probs = logits.softmax(dim=-1)
    print(f"\n{name}:")
    for desc, prob in zip(descriptions, probs):
        print(f"  {prob:.3f} — {desc}")

## Audio-to-Audio Similarity

Find similar audio by comparing embeddings.

In [ ]:
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine

# Compute audio embeddings for all samples
embeddings = {}
for name, audio in samples.items():
    inputs = processor(audios=[audio], sampling_rate=sr, return_tensors="pt", padding=True)
    with torch.no_grad():
        emb = model.get_audio_features(**inputs)
    embeddings[name] = emb[0].numpy()

# Compute pairwise similarity matrix
names = list(embeddings.keys())
n = len(names)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = 1 - cosine(embeddings[names[i]], embeddings[names[j]])

# Visualize
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_yticklabels(names)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{sim_matrix[i,j]:.2f}", ha='center', va='center')
plt.colorbar(im, label='Cosine Similarity')
plt.title('Audio-to-Audio Similarity (CLAP Embeddings)')
plt.tight_layout()
plt.show()

## Building a Simple Music Search Engine

Use CLAP to search a collection of audio files by text query.

In [ ]:
from scipy.spatial.distance import cosine

def search_by_text(query, audio_collection, audio_embeddings, top_k=3):
    """Search audio collection using a text query via CLAP embeddings."""
    # Embed the text query
    inputs = processor(text=[query], return_tensors="pt", padding=True)
    with torch.no_grad():
        text_emb = model.get_text_features(**inputs)[0].numpy()
    
    # Rank by cosine similarity
    scores = []
    for name, audio_emb in audio_embeddings.items():
        sim = 1 - cosine(text_emb, audio_emb)
        scores.append((name, sim))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

# Demo queries
queries = [
    "a steady drumbeat",
    "a pure musical tone",
    "a harmonic chord progression",
    "electronic music with bass",
]

for query in queries:
    results = search_by_text(query, samples, embeddings)
    print(f"\nQuery: '{query}'")
    for name, score in results:
        print(f"  {score:.3f} — {name}")

## How Recommender Systems Use Embeddings

Spotify, Apple Music, and YouTube Music use similar embedding models to:
1. Find songs similar to what you're listening to
2. Build 'radio stations' from seed tracks
3. Create personalized playlists
4. Cluster music by mood/genre automatically